In [1]:
import numpy as np
import pandas as pd
import sklearn

In [2]:
print("Numpy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Numpy: 2.5.4
Pandas: 3.1.0
scikit-learn: 1.9.1


In [3]:
from datasets import load_dataset

In [4]:
dataset = load_dataset("fancyzhx/ag_news")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [11]:
example = dataset["train"][0]

print("Article:")
print(example["text"])

print("\nLabel:")
print(example["label"])

Article:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Label:
2


### Examine class distribution

In [14]:
train_df = dataset["train"].to_pandas()
print(train_df["label"].value_counts().sort_index())

label
0    30000
1    30000
2    30000
3    30000
Name: count, dtype: int64


we have checked training set so far. We will also verify the test set distribution later

In [15]:
for i in range(5):
    print(f"\nArticle {i + 1}")
    print("Text:", dataset["train"][i]["text"])
    print("Label:", dataset["train"][i]["label"])


Article 1
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Label: 2

Article 2
Text: Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
Label: 2

Article 3
Text: Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
Label: 2

Article 4
Text: Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructure, an oil official said on Saturday.
Lab

In [26]:
texts = train_df["text"]

backslash_count = texts.str.contains(r"\\", regex=True).sum()

print("Articles containing a backslash:", backslash_count)
print("Total training articles:", len(texts))
print(f"Percentage: {round(backslash_count / len(texts) * 100, 2)} %")

Articles containing a backslash: 13146
Total training articles: 120000
Percentage: 10.96 %


In [ ]:
backslash_articles = train_df[
    train_df["text"].str.contains(r"\\", regex=True)
]

for i in range(5):
    print(f"\nArticle {i + 1}:")
    print(backslash_articles.iloc[i]["text"])


Article 1:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Article 2:
Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.

Article 3:
Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.

Article 4:
Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructure, an oil official said on Saturday.

Article 5:
Stocks End Up, But Near Year Lows (Reu

In [32]:
sample_texts = backslash_articles["text"].head(5)

for text in sample_texts:
    cleaned_text = text.replace("\\", " ")

    print(f"Original: {text}")
    print(f"Cleaned: {cleaned_text}")
    print("-" * 80)

Original: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Cleaned: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.
--------------------------------------------------------------------------------
Original: Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
Cleaned: Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group, which has a reputation for making well-timed and occasionally controversial plays in the defense industry, has quietly placed its bets on another part of the market.
---------------------------------------------------------

In [33]:
train_df["cleaned_text"] = train_df["text"].str.replace("\\", " ", regex=False)

In [40]:
print("Original:")
print(train_df.loc[0, "text"])

print("\nCleaned:")
print(train_df.loc[0, "cleaned_text"])

Original:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Cleaned:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.


- **Observation**: Some news articles contain backslashes in the middle of words or sentences.
- **Hypothesis**: These backslashes may be artifacts from the dataset's text formatting rather than meaningful parts of the news articles.
- **Experiment**: Create a separate `cleaned_text` column by replacing literal backslashes with spaces, while preserving the original text
- **Current result**: The example look more readable after cleaning

**We dont have demonstrated that cleaning improves classification performance**

In [41]:
print(
    f"backslashed remaining: {train_df["cleaned_text"].str.contains("\\", regex=False).sum()}"
)

backslashed remaining: 0


We found backslashes in 10.96% of training examples, where these backslashing might be just formatting artifacts. Our experiment is to replace each backslash with a space while preserving the original text. we will test it whether this effects the model performance

### Check Test Dataset

In [50]:
test_df = dataset["test"].to_pandas()

print(f"Test dataset shape: {test_df.shape}")

print("\nClass Distribution:")
print(test_df["label"].value_counts().sort_index())

print("\nArticles containing backslashes:")
backslash_count = test_df["text"].str.contains("\\", regex=False).sum()

print(backslash_count)
print(f"Percentage: {round(backslash_count / len(test_df) * 100, 2)}%")

Test dataset shape: (7600, 2)

Class Distribution:
label
0    1900
1    1900
2    1900
3    1900
Name: count, dtype: int64

Articles containing backslashes:
808
Percentage: 10.63%


Since we created `cleaned_text` for training, lets do it for testing as well

In [55]:
test_df["cleaned_text"] = test_df["text"].str.replace("\\", " ", regex=False)

print("Original:")
print(test_df.loc[11, "text"])

print("\nCleaned:")
print(test_df.loc[11, "cleaned_text"])

print(
    f"\nBackslashes remaining: {test_df["cleaned_text"].str.contains("\\", regex=False).sum()}"
)

Original:
Apple Launches Graphics Software, Video Bundle  LOS ANGELES (Reuters) - Apple Computer Inc.&lt;AAPL.O&gt; on  Tuesday began shipping a new program designed to let users  create real-time motion graphics and unveiled a discount  video-editing software bundle featuring its flagship Final Cut  Pro software.

Cleaned:
Apple Launches Graphics Software, Video Bundle  LOS ANGELES (Reuters) - Apple Computer Inc.&lt;AAPL.O&gt; on  Tuesday began shipping a new program designed to let users  create real-time motion graphics and unveiled a discount  video-editing software bundle featuring its flagship Final Cut  Pro software.

Backslashes remaining: 0


The news articles contain `&lt`; and `&gt` sequences, they are HTML entities, we also have multiple spaces between words, **but** i would not start changing eveything at once-that would make it harder to understand which change actually hepls our model. I think that these features carry useful information in news articles. We should not assume that removing the entities helps, nor should we assume keeping them is always better

**Lets leave the HTML entities untouched for now**

In [58]:
html_entities = {
    "&lt;": "Less-than sign",
    "&gt;": "Greater-than sign",
    "&amp;": "Ampersand",
    "&quot;": "Quotation mark",
}

for entity, description in html_entities.items():
    count = train_df["text"].str.contains(entity, regex=False).sum()

    percentage = count / len(train_df) * 100

    print(
        f"{description} ({entity}): "
        f"{count} articles ({percentage:.2f}%)"
    )

Less-than sign (&lt;): 5241 articles (4.37%)
Greater-than sign (&gt;): 5243 articles (4.37%)
Ampersand (&amp;): 0 articles (0.00%)
Quotation mark (&quot;): 0 articles (0.00%)


**about 4.37% of training contain each of the less-than and grate-than entities.** These could appear together in the same artile, to represent a stock ticker

### Splitting into train/dev/test

In [60]:
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (120000, 3)
Test shape: (7600, 3)


AG news already provides training and testing sets, so we split the existing **120,000 training examples** into training and dev sets, keeping the original test articles untouched. So we do 80/20 split of the original training set

### Data Exploration Summary

1. **Class Distribution**
i examined the class distribution in both the training and test sets. Each of the four classes contains **30,000 training samples and 1,900 test samples**. Therefore, both datasets are perfectly balanced, with each class **representing 25%** of its respective datasets

2. **Text Inspection and Preprocessing Decisions**
While inspecting the news articles, I noticed backslashes and HTML entities in the text. I decided to preserve the HTML entities for now because I have not yet established whether modifying them would improve classification performance.

3. **Backslash Analysis**
Before cleaning the text, I calculated the percentage of articles containing backslashes:

    Training set: 10.96%

    Test set: 10.63%

Since backslashes appeared in both datasets at similar rates, I decided to replace them with spaces as an initial preprocessing step.

4. **Preserving the Original Text**
I created a separate cleaned_text column in each DataFrame rather than modifying the original text column. This allows me to compare the original and cleaned versions in future experiments.

5. **HTML Entity Analysis**
I calculated the frequency and percentage of training articles containing several common HTML entities, including `&lt;`, `&gt;`, `&amp;`, and `&quot;`. The results showed that `&lt;` appeared in **4.37%** of articles and `&gt;` appeared in **4.37%**. The other two entities were not found.

For now, I will leave these entities unchanged and revisit this decision if future experiments suggest that additional preprocessing may be beneficial.